# Zaawansowane Skalowanie Obrazu: Interpolacja Dwusześcienna (Bicubic)

Niniejszy notatnik prezentuje w pełni autorską, niskopoziomową implementację algorytmu interpolacji dwusześciennej. 

W przeciwieństwie do interpolacji dwuliniowej (analizującej 4 najbliższych sąsiadów), metoda dwusześcienna przetwarza siatkę 16 pikseli (otoczenie 4x4) wokół punktu subpikselowego. Pozwala to na znacznie dokładniejszą aproksymację funkcji ciągłej rozkładu jasności, redukując efekt "schodkowania" (aliasingu) i zachowując wyższą ostrość krawędzi.

### Architektura i Ochrona Pamięci (Edge Cases)
Z inżynierskiego punktu widzenia, największym wyzwaniem przy analizie sąsiedztwa 4x4 jest obsługa skrajnych krawędzi obrazu (brzegów macierzy). W poniższej implementacji zastosowano profesjonalną technikę zabezpieczającą: **padding z replikacją brzegów** (`cv2.copyMakeBorder`). Rozszerza ona wirtualnie pamięć obrazu, co zapobiega powstawaniu wyjątków typu *Index Out of Bounds* podczas estymacji pochodnych cząstkowych ($A_x, A_y, A_{xy}$) na granicach matrycy.

### Złożoność Obliczeniowa (Trade-off)
Rozwiązanie to stanowi klasyczny kompromis architektoniczny (trade-off) pomiędzy jakością a wydajnością. Wymaga wyznaczenia 16 współczynników wielomianu dla każdego interpolowanego piksela poprzez operacje mnożenia wektorów przez prekompilowaną macierz odwrotną ($A^{-1}$), co znacząco obciąża jednostkę obliczeniową w porównaniu do metod liniowych.

In [ ]:
import cv2
import os
from matplotlib import pyplot as plt
import numpy as np

# Pobieranie wymaganych plików źródłowych i prekompilowanej macierzy
if not os.path.exists("parrot.bmp") :
    !wget https://raw.githubusercontent.com/vision-agh/poc_sw/master/05_Resolution/parrot.bmp --no-check-certificate
if not os.path.exists("clock.bmp") :
    !wget https://raw.githubusercontent.com/vision-agh/poc_sw/master/05_Resolution/clock.bmp --no-check-certificate
if not os.path.exists("chessboard.bmp") :
    !wget https://raw.githubusercontent.com/vision-agh/poc_sw/master/05_Resolution/chessboard.bmp --no-check-certificate

if not os.path.exists("ainvert.py") :
    !wget https://raw.githubusercontent.com/vision-agh/poc_sw/master/05_Resolution/ainvert.py --no-check-certificate

import ainvert

def bicubic_interpolation(img, scale_factor):
    img_float = img.astype(np.float32)
    
    h, w = img_float.shape[:2]
    new_h, new_w = int(h * scale_factor), int(w * scale_factor)
    new_img = np.zeros((new_h, new_w), dtype=np.float32)
    
    # Zabezpieczenie przed wyjściem poza zakres dla skrajnych pikseli (Padding)
    pad_img = cv2.copyMakeBorder(img_float, 2, 2, 2, 2, cv2.BORDER_REPLICATE)
    
    A_inv = ainvert.A_invert
    
    for i in range(new_h):
        for j in range(new_w):
            x = i / scale_factor
            y = j / scale_factor
            
            dx = x - np.floor(x)
            dy = y - np.floor(y)
            
            # Przesunięcie indeksów uwzględniające dodany padding
            i0 = int(np.floor(x)) + 2 - 1
            j0 = int(np.floor(y)) + 2 - 1

            neighborhood = pad_img[i0:i0+4, j0:j0+4]

            # Pobranie węzłów macierzy
            A = neighborhood[1, 1]
            B = neighborhood[2, 1]
            C = neighborhood[2, 2]
            D = neighborhood[1, 2]

            # Estymacja pochodnych cząstkowych
            Ax = (neighborhood[2, 1] - neighborhood[0, 1]) / 2.0
            Ay = (neighborhood[1, 2] - neighborhood[1, 0]) / 2.0
            Axy = (neighborhood[2, 2] - neighborhood[0, 1] - neighborhood[1, 0] + neighborhood[1, 1]) / 4.0
            
            Bx = (neighborhood[3, 1] - neighborhood[1, 1]) / 2.0
            By = (neighborhood[2, 2] - neighborhood[2, 0]) / 2.0
            Bxy = (neighborhood[3, 2] - neighborhood[1, 1] - neighborhood[2, 0] + neighborhood[2, 1]) / 4.0
            
            Cx = (neighborhood[3, 2] - neighborhood[1, 2]) / 2.0
            Cy = (neighborhood[2, 3] - neighborhood[2, 1]) / 2.0
            Cxy = (neighborhood[3, 3] - neighborhood[1, 2] - neighborhood[2, 1] + neighborhood[2, 2]) / 4.0
            
            Dx = (neighborhood[2, 2] - neighborhood[0, 2]) / 2.0
            Dy = (neighborhood[1, 3] - neighborhood[1, 1]) / 2.0
            Dxy = (neighborhood[2, 3] - neighborhood[0, 2] - neighborhood[1, 1] + neighborhood[1, 2]) / 4.0
            
            x_vec = np.array([A, B, D, C, Ax, Bx, Dx, Cx, Ay, By, Dy, Cy, Axy, Bxy, Dxy, Cxy])
            
            # Rozwiązanie układu równań liniowych
            a = A_inv @ x_vec
            
            value = 0.0
            idx = 0
            for q in range(4):     
                for p in range(4): 
                    value += a[idx] * (dx ** p) * (dy ** q)
                    idx += 1
            
            new_img[i, j] = value
            
    return np.clip(new_img, 0, 255).astype(np.uint8)

img = cv2.imread('parrot.bmp', cv2.IMREAD_GRAYSCALE)

scale = 1.5
result = bicubic_interpolation(img, scale)

plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.imshow(img, cmap='gray')
plt.title('Oryginał')
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(result, cmap='gray')
plt.title(f'Interpolacja Dwusześcienna (Skala: {scale}x)')
plt.axis('off')

plt.tight_layout()
plt.show()

### Czyszczenie przestrzeni roboczej

In [ ]:
import shutil
import glob

trash_files = glob.glob('*.bmp*') + glob.glob('ainvert.py*')

for file in trash_files:
    try:
        os.remove(file)
    except OSError:
        pass

if os.path.exists('__pycache__'):
    shutil.rmtree('__pycache__')

print("Środowisko oczyszczone z plików tymczasowych.")